# Inverse Translation: optimize discrete inputs

Provided FSMT + target token IDs → verify four provided examples → stream vocabulary probes → coordinate substitutions minimizing target decision margin → verify actual generated tokens → official evaluator.

**Điểm đã đo ở các version Kaggle trước:** R06 official100 exact24/24; Kaggle100/100; code11.971s,7832 candidate evaluations. Gradient R08/R09 có code nghiên cứu, kết quả xem STATUS mới nhất.

**Bài học:** Giữ token IDs của model, không thay BPE khác. Model frozen là một phần bắt buộc của đề. Chỉ gọi tiết kiệm khi giữ exact score; tham khảo HotFlip và AutoPrompt cho RUN_ID8/9.

Notebook này chứa source đầy đủ nhưng chưa execute tại bản export này; execution_count=null và outputs rỗng là có chủ ý. Không suy diễn metric mới từ notebook chưa chạy.

Bật GPU T4 trên Kaggle, attach competition tương ứng trước Run All. Loader tự tìm cả `/kaggle/input/competitions/<slug>` và `/kaggle/input/<slug>`. Code assert T4, không tự chuyển CPU. Thực hành sau contest, không dùng trong thời gian cuộc thi cấm trợ giúp.


# Inverse Translation: tối ưu input rời rạc ở cùng mức điểm

Target text/token IDs và FSMT được đề cung cấp. Đây là tìm input cho frozen model, không phải train dịch thuật hay tự suy ra private target. R01/R04/R05/R06/R07 đều đạt100 public/private. R06 code11.97s so R01 16.29s trong một batch; warm-up khiến so thời gian một lần chỉ mang tính tham khảo. R02 top8 hụt hai target, R03 top16 hụt một target: bottleneck là recall của proposal từ probe một token, không chỉ số vòng search.

| Hạng | Paper | Phần áp dụng | Điều không tái lập |
|---|---|---|---|
| 1 | [HotFlip, Ebrahimi et al., ACL 2018](https://aclanthology.org/P18-2006.pdf) §3 | Xấp xỉ thay token bằng gradient theo input embedding, shortlist rồi exact evaluate | Task này là puzzle trên model cung cấp, không perturb văn bản người dùng. Dùng token-ID đúng vocab nguồn |
| 2 | [AutoPrompt, Shin et al., EMNLP 2020](https://aclanthology.org/2020.emnlp-main.346.pdf) §2.2 | Gradient-guided candidate generation thay vì scan mọi token với model forward | Không dùng MLM/BERT khác; không tối ưu universal prompt cho dataset khác |
| 3 | [Universal Adversarial Triggers, Wallace et al., EMNLP 2019](https://aclanthology.org/D19-1221.pdf) §2 | Tối ưu token rời rạc, exact search sau first-order ranking | Không giả định cùng một trigger giải được24 target; mỗi row có input riêng |
| 4 | [Automatically Auditing Language Models via Discrete Optimization / ARCA, Jones et al., ICML 2023](https://proceedings.mlr.press/v202/jones23a/jones23a.pdf) §3 | Objective gần hành vi greedy và kiểm tra exact top-k, giảm sai lệch surrogate | Chưa cài averaged-gradient ARCA hay joint output optimization; target cố định nên không sửa target |
| 5 | [Hard Prompts Made Easy, Wen et al., NeurIPS 2023](https://proceedings.neurips.cc/paper_files/paper/2023/hash/a00548031e4647b13042c97c922fadf1-Abstract-Conference.html) | Projected embedding optimization là phương án nếu local token swaps bị kẹt | Chưa cài PEZ; chỉ đọc abstract/source mô tả, PDF lớn không tải được qua retrieval. Không tuyên bố review đầy đủ thuật toán |

**R08:** dùng top8 grid search nhanh như R02. Với row chưa exact, backprop CE của target qua frozen FSMT tới embedding input tạm thời. Rank 23221 token bằng −grad·embedding; lấy16 token/position; exact-evaluate decision-margin của các phép thay. Chỉ chấp nhận nếu margin giảm. Max16 gradient steps. Nếu còn sai, fallback shortlist probe được ghi vào chi phí.

**R09:** khởi tạo từ top1 probe và dùng gradient substitutions ngay; tối đa24 bước. Dù ý tưởng thuộc họ coordinate optimization, code đánh giá toàn bộ single-position proposals rồi thay một vị trí tốt nhất mỗi bước, không phải sweep cập nhật mọi vị trí. Dùng cùng fallback top8.

Giữ canary4 examples, Transformers4.57.1, hook vị trí decoder đã kiểm, bỏ hook trước official evaluator. Không sửa model weights. Báo exact24/24, official score, gradient calls, model calls, candidate evaluations, walltime, peakVRAM. Chỉ kết luận tiết kiệm nếu vẫn100; nếu không thì ghi tradeoff, không gọi nhanh hơn là tốt hơn.


In [ ]:
SOURCE_COMMIT = '84e9a9efaa6346578162fa12915a8368f18e6485'
RUN_ID = 6
import subprocess, sys
subprocess.run([sys.executable,"-m","pip","install","-q","transformers==4.57.1"],check=True)


## Shared utilities

Seed, BPE, masking, data paths, schema checks và report writer.


In [ ]:
"""Utilities for controlled, idea-driven AICC NLP experiments."""
import ast, csv, hashlib, importlib.metadata, json, math, os, random, time
from copy import deepcopy
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders

SEED = 42
OUT = Path("/kaggle/working")
DEVICE = torch.device("cuda:0")


def start_run(task, run_id, config):
    assert torch.cuda.is_available(), "GPU T4 is required; do not silently fall back to CPU"
    name = torch.cuda.get_device_name(0)
    assert "T4" in name, "Expected T4, received " + name
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.set_num_threads(2)
    torch.backends.cudnn.benchmark = False
    torch.cuda.reset_peak_memory_stats()
    info = dict(task=task, run_id=f"R{run_id:02d}", config=config, seed=SEED,
                device=name, gpu_count=torch.cuda.device_count(), started=time.time(),
                source_commit=globals().get("SOURCE_COMMIT"),
                versions={p: importlib.metadata.version(p) for p in
                          ["torch", "numpy", "pandas", "scikit-learn", "tokenizers"]})
    print("RUN_START " + json.dumps(info), flush=True)
    return info


def find_data(slug):
    for root in (Path("/kaggle/input/competitions")/slug, Path("/kaggle/input")/slug):
        if root.is_dir():
            return root
    raise FileNotFoundError("Attach competition data: " + slug)


def make_bpe(texts, size=2048):
    tokenizer = Tokenizer(models.BPE(unk_token="[UNK]"))
    tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()
    tokenizer.decoder = decoders.BPEDecoder()
    tokenizer.train_from_iterator(texts, trainers.BpeTrainer(
        vocab_size=size, min_frequency=2,
        special_tokens=["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"]))
    return tokenizer


def encode_texts(tokenizer, texts, max_length):
    values = np.zeros((len(texts), max_length), dtype=np.int64)
    for i, encoded in enumerate(tokenizer.encode_batch(texts)):
        ids = encoded.ids[:max_length] or [1]
        values[i, :len(ids)] = ids
    return values


def hash_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def finish_run(info, validation, submission, checks, history, artifacts=None, limitations=None):
    assert checks.get("ids_unique") is True and checks.get("schema_valid") is True
    assert checks.get("row_count", 0) > 0
    submission.to_csv(OUT/"submission.csv", index=False)
    info.update(status="completed", elapsed_seconds=time.time()-info.pop("started"),
                validation=validation, submission_checks=checks, history=history,
                peak_gpu_memory_mb=torch.cuda.max_memory_allocated()/1024**2,
                artifacts=artifacts or {}, limitations=limitations or [],
                leaderboard_public=None, leaderboard_private=None)
    (OUT/"run_report.json").write_text(json.dumps(info, indent=2), encoding="utf-8")
    print("NLP_RUN_RESULT " + json.dumps(info), flush=True)
    print("NLP_RUN_PASSED", flush=True)
    return info


class PositionEncoder(nn.Module):
    def __init__(self, vocab, dim=96, layers=2, max_length=384, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab, dim, padding_idx=0)
        self.position = nn.Embedding(max_length, dim)
        block = nn.TransformerEncoderLayer(dim, 4, dim*3, dropout=dropout,
                                           activation="gelu", batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(block, layers, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(dim)

    def forward(self, ids):
        mask = ids.eq(0)
        x = self.embedding(ids) + self.position(torch.arange(ids.shape[1], device=ids.device))
        x = self.encoder(x, src_key_padding_mask=mask)
        return self.norm(x), mask


def pools(hidden, mask):
    valid = (~mask).unsqueeze(-1)
    mean = (hidden*valid).sum(1)/valid.sum(1).clamp(min=1)
    maximum = hidden.masked_fill(~valid, -1e4).amax(1)
    return torch.cat([mean, maximum], -1)



## Model và experiments

Các config Rxx nằm đầu source. Đổi RUN_ID trước khi Run All; mỗi lượt phải bắt đầu model/optimizer mới.


In [ ]:
"""Inverse Translation: memory-bounded probing and controlled discrete search."""
# Organizer reference by cowile, AI-Community-Contest/solutions (MIT).
import html, re, sys
from transformers import FSMTForConditionalGeneration

INVERSE_CONFIGS={
 1:dict(idea="Streaming top-32 probe avoids materializing the vocabulary product",topk=32,loss="margin",search="grid",restart=0),
 2:dict(idea="Top-8 proposal set trades candidate recall for speed",topk=8,loss="margin",search="grid",restart=0),
 3:dict(idea="Top-16 measures the middle recall-cost tradeoff",topk=16,loss="margin",search="grid",restart=0),
 4:dict(idea="Teacher-forced CE instead of decision margin",topk=32,loss="ce",search="grid",restart=0),
 5:dict(idea="Coarse-to-fine proposal search top-8 then top-32",topk=32,loss="margin",search="coarse",restart=0),
 6:dict(idea="Coordinate sweeps replace the full substitution grid",topk=32,loss="margin",search="coordinate",restart=0),
 7:dict(idea="Selective restarts spend compute only on non-exact rows",topk=32,loss="margin",search="coarse",restart=2)
}
INVERSE_CONFIGS[8]=dict(idea="Top8 exact search with contextual HotFlip fallback",topk=8,loss="margin",search="grid",restart=0,gradient=True,gradient_topk=16,gradient_steps=16)
INVERSE_CONFIGS[9]=dict(idea="Contextual gradient coordinate search from top1 probe seeds",topk=8,loss="margin",search="gradient",restart=0,gradient=True,gradient_topk=16,gradient_steps=24)


def run_inverse(run_id):
    cfg={**INVERSE_CONFIGS[run_id],"probe_batch":512,"candidate_batch":64,"budget_seconds":720}
    info=start_run("inverse-translation",run_id,cfg)
    info["versions"]["transformers"]=importlib.metadata.version("transformers")
    root=find_data("inverse-translation-aicc-round-10")
    targets=json.loads((root/"targets.json").read_text())
    examples=json.loads((root/"examples.json").read_text())
    english=json.loads((root/"english_vocab.json").read_text())
    model=FSMTForConditionalGeneration.from_pretrained(root/"model").to(DEVICE).eval()
    for p in model.parameters():p.requires_grad=False
    def decode(ids):
        text="".join(english.get(str(i)," ? ") for i in ids)
        text=html.unescape(text.replace("</s>","").replace("</w>"," "))
        text=re.sub(" (?=[.,?!;:])",""," ".join(text.split()))
        return re.sub(" ?' ?","'",text)
    @torch.inference_mode()
    def generate(source,max_new):
        source_ids=F.pad(source[None],(0,1),value=2)
        output=model.generate(input_ids=source_ids,num_beams=1,max_new_tokens=max_new)[0].cpu().tolist()
        ids=[x for x in output if x!=1]
        return ids[1:] if ids and ids[0]==2 else ids
    # Verify the unmodified provided model before adapting teacher-forced positions.
    example_matches=[]
    for row in examples:
        got=generate(torch.tensor(row["alien_ids"],device=DEVICE),row["target_content_tokens"]+16)
        example_matches.append(got==row["target_ids"])
        print("EXAMPLE_CHECK "+json.dumps(dict(match=example_matches[-1],expected=row["target_ids"],generated=got,expected_text=decode(row["target_ids"]),generated_text=decode(got))),flush=True)
    if not all(example_matches):
        raise RuntimeError("Provided model generation does not match the provided examples")
    def cached_positions(module,inputs,positions):
        return positions[:,:1].expand_as(positions)
    handle=model.model.decoder.embed_positions.register_forward_hook(cached_positions)
    counters=dict(probe_batches=0,candidate_evaluations=0,model_calls=0,gradient_calls=0)
    @torch.inference_mode()
    def probe():
        values=None;indices=None;tick=time.time()
        for batch in torch.arange(3,23224,device=DEVICE).split(cfg["probe_batch"]):
            source=F.pad(batch[:,None],(0,1),value=2)
            starts=torch.full((len(batch),1),2,device=DEVICE,dtype=torch.long)
            logits=model(input_ids=source,decoder_input_ids=starts,use_cache=False).logits[:,0]
            score=logits.float().log_softmax(-1).T
            local_values,local_indices=score.topk(min(cfg["topk"],len(batch)),dim=1)
            local_ids=batch[local_indices]
            if values is None:values,indices=local_values,local_ids
            else:
                joined=torch.cat([values,local_values],dim=1)
                joined_ids=torch.cat([indices,local_ids],dim=1)
                values,positions=joined.topk(cfg["topk"],dim=1)
                indices=joined_ids.gather(1,positions)
            counters["probe_batches"]+=1
            del logits,score
        return indices,time.time()-tick
    dictionary,probe_seconds=probe()
    @torch.inference_mode()
    def losses(sources,target):
        costs=[]
        for batch in sources.split(cfg["candidate_batch"]):
            labels=target.expand(len(batch),-1)
            source=F.pad(batch,(0,1),value=2)
            decoder=F.pad(labels[:,:-1],(1,0),value=2)
            logits=model(input_ids=source,decoder_input_ids=decoder,use_cache=False).logits.float()
            wanted=logits.gather(2,labels[...,None]).squeeze(-1)
            if cfg["loss"]=="margin":cost=(logits.amax(-1)-wanted).mean(1)
            else:cost=(logits.logsumexp(-1)-wanted).mean(1)
            costs.append(cost)
            counters["candidate_evaluations"]+=len(batch);counters["model_calls"]+=1
        return torch.cat(costs)
    def substitutions(source,choices):
        length,k=choices.shape
        candidates=source.repeat(length*k,1)
        rows=torch.arange(length*k,device=DEVICE)
        positions=torch.arange(length,device=DEVICE).repeat_interleave(k)
        candidates[rows,positions]=choices.flatten()
        return candidates
    def contextual_choices(source,target):
        # Gradients flow only to a temporary input embedding; all model weights stay frozen.
        captured={}
        def leaf_embedding(module,inputs,output):
            leaf=output.detach().requires_grad_(True);captured['leaf']=leaf
            return leaf
        h=model.model.encoder.embed_tokens.register_forward_hook(leaf_embedding)
        try:
            with torch.enable_grad():
                src=F.pad(source[None],(0,1),value=2)
                decoder=F.pad(target[None,:-1],(1,0),value=2)
                logits=model(input_ids=src,decoder_input_ids=decoder,use_cache=False).logits.float()
                objective=F.cross_entropy(logits.reshape(-1,logits.shape[-1]),target)
                gradient=torch.autograd.grad(objective,captured['leaf'])[0][0,:len(source)].detach()
            assert torch.isfinite(gradient).all() and gradient.abs().sum()>0
            weight=model.model.encoder.embed_tokens.weight.detach()
            gains=-(gradient@weight[3:23224].T)
            choices=gains.topk(cfg['gradient_topk'],dim=-1).indices+3
            counters['gradient_calls']+=1;counters['model_calls']+=1
            return choices
        finally:h.remove()
    def gradient_search(source,target):
        source=source.clone();current=float(losses(source[None],target)[0])
        for step in range(cfg['gradient_steps']):
            if current<=1e-7 or time.time()-info['started']>cfg['budget_seconds']:break
            choices=contextual_choices(source,target)
            candidates=substitutions(source,choices)
            costs=losses(candidates,target);best=int(costs.argmin());value=float(costs[best])
            if value>=current-1e-7:break
            source,current=candidates[best],value
        return source,current
    def search(target,choices,source=None):
        source=choices[:,0].clone() if source is None else source.clone()
        current=float(losses(source[None],target)[0])
        for step in range(80):
            if time.time()-info["started"]>cfg["budget_seconds"]:break
            improved=False
            if cfg["search"]=="coordinate":
                for pos in range(len(source)):
                    candidates=source.repeat(choices.shape[1],1)
                    candidates[:,pos]=choices[pos]
                    costs=losses(candidates,target);best=int(costs.argmin());value=float(costs[best])
                    if value<current-1e-7:source,current=candidates[best],value;improved=True
            else:
                candidates=substitutions(source,choices)
                costs=losses(candidates,target);best=int(costs.argmin());value=float(costs[best])
                if value<current-1e-7:source,current=candidates[best],value;improved=True
            if not improved or (cfg["loss"]=="margin" and current<=1e-7):break
        return source,current
    solved={};history=[]
    for row in targets:
        target=torch.tensor(row["target_ids"],device=DEVICE)
        choices=dictionary[target[:-1]]
        if cfg['search']=='gradient':
            source,cost=gradient_search(choices[:,0],target)
        elif cfg["search"]=="coarse":
            source,cost=search(target,choices[:,:min(8,choices.shape[1])])
            source,cost=search(target,choices,source)
        else:source,cost=search(target,choices)
        generated=generate(source,row["target_content_tokens"]+16)
        if cfg.get('gradient') and generated!=row['target_ids']:
            source,cost=gradient_search(source,target)
            generated=generate(source,row['target_content_tokens']+16)
            # A frozen probe shortlist remains an explicit fallback, never hidden in cost reporting.
            if generated!=row['target_ids']:
                source,cost=search(target,choices,source)
                generated=generate(source,row['target_content_tokens']+16)
        for attempt in range(cfg["restart"]):
            if generated==row["target_ids"]:break
            random_indices=torch.randint(0,choices.shape[1],(len(source),),device=DEVICE)
            proposal=choices[torch.arange(len(source),device=DEVICE),random_indices]
            candidate,new_cost=search(target,choices,proposal)
            if new_cost<cost:
                source,cost=candidate,new_cost
                generated=generate(source,row["target_content_tokens"]+16)
        ids=source.cpu().tolist()
        assert 1<=len(ids)<=row["max_source_length"] and all(3<=x<=23223 for x in ids)
        solved[row["row_id"]]=ids
        record=dict(row_id=row["row_id"],exact=generated==row["target_ids"],
                    loss=cost,generated_text=decode(generated),source_length=len(ids))
        history.append(record);print("TARGET "+json.dumps(record),flush=True)
    handle.remove()
    sub=pd.DataFrame(dict(row_id=sorted(solved),source_ids=[json.dumps(solved[i]) for i in sorted(solved)]))
    sub.to_csv(OUT/"submission.csv",index=False)
    del model;torch.cuda.empty_cache()
    sys.path.insert(0,str(root))
    from evaluation import score_submission
    result=score_submission(root,OUT/"submission.csv")
    result={k:float(v) for k,v in result.items() if k!="rows"}
    checks=dict(ids_unique=sub.row_id.is_unique,schema_valid=list(sub.columns)==["row_id","source_ids"],
                row_count=len(sub),token_ranges_valid=True,lengths_valid=True)
    return finish_run(info,dict(metric="official_score_0_100",value=result["score"],direction="maximize",
          samples=len(targets),split="explicit_public_targets",official=result),sub,checks,history,
          artifacts=dict(probe_seconds=probe_seconds,**counters,checkpoint="provided FSMT only",
                         targets_sha256=hash_file(root/"targets.json")),
          limitations=["The target outputs are explicitly supplied by the task; this is input optimization.",
                       "Reference already reaches 100; compare cost and robustness at matched score.",
                       "Preserve provided token IDs; an unrelated BPE tokenizer would invalidate the model."])



In [ ]:
OUT = Path('/kaggle/working') / 'inverse-translation' / f'R{RUN_ID:02d}'
OUT.mkdir(parents=True, exist_ok=True)
report = run_inverse(RUN_ID)
print('Submission:', OUT / 'submission.csv')
